In [ ]:
# @title
import os
import pandas as pd
import matplotlib.pyplot as pl
import numpy as np
import math
import sys
import scipy
from scipy.optimize import curve_fit
from scipy.stats import linregress as lr
print(np.__version__)
print(pd.__version__)
print(scipy.__version__)
import matplotlib
print(matplotlib.__version__)

from google.colab import drive
drive.mount('/content/drive',force_remount=True)
base='/content/drive/MyDrive/Modelo/eATP-kinetics-model'
#base='*/eATP-kinetics-model'

2.1.3
2.2.3
1.16.3
3.10.0
Mounted at /content/drive


In [ ]:
# @title
ecto_Biwo=pd.read_csv(base+'/data/Activities/BeWo.csv',index_col=1)
ecto_RBC=pd.read_csv(base+'/data/Activities/RBC.csv',index_col=1)

masa_cel=200 #ug
print('Cell mass employed ',masa_cel,' ug')

k_h_s_e=ecto_Biwo['Sincicio'].loc['Pendiente'] # nM/(min*ug*nM ATP)
k_h_c_e=ecto_Biwo['Cito'].loc['Pendiente'] # nM/(min*ug*nM ATP)

k_h_s=k_h_s_e*200 # nM hidrolizado/(nM ATP*min)
k_h_c=k_h_c_e*200 # nM hidrolizado/(nM ATP*min)
print(ecto_RBC)
k_h_irbc=-ecto_RBC['Value'].loc['iRBC_slope']
k_h_crbc=-ecto_RBC['Value'].loc['cRBC_slope']

units=' nM hydrolized/(nM ATP*min)'

print('Condition NF54 k_hydrolisis: ',k_h_irbc+k_h_c,units)
print('Condition FCR3 k_hydrolisis: ',k_h_irbc+k_h_c,units)
print('Condition cRBCs k_hydrolisis: ',k_h_crbc+k_h_c,units)
print('Condition Mechanical stimulus k_hydrolisis: ',k_h_c,units)

Cell mass employed  200  ug
                Unnamed: 0     Value
Condition                           
iRBC_slope               0 -0.001554
iRBC_intercept           1 -0.055251
cRBC_slope               2 -0.000213
cRBC_intercept           3 -0.048202
Condition NF54 k_hydrolisis:  0.0160341503341318  nM hydrolized/(nM ATP*min)
Condition FCR3 k_hydrolisis:  0.0160341503341318  nM hydrolized/(nM ATP*min)
Condition cRBCs k_hydrolisis:  0.0146932295520491  nM hydrolized/(nM ATP*min)
Condition Mechanical stimulus k_hydrolisis:  0.01448  nM hydrolized/(nM ATP*min)


In [ ]:
data_base=base+'/data/Datos.xlsx'
data_nf54=pd.read_excel(data_base,sheet_name='Data',skiprows=1,usecols="B:E",nrows=7)
data_fcr3=pd.read_excel(data_base,sheet_name='Data',skiprows=1,usecols="F:I",nrows=7)
data_rbcs=pd.read_excel(data_base,sheet_name='Data',skiprows=1,usecols="J:M",nrows=7)
data_biwo=pd.read_excel(data_base,sheet_name='Data',skiprows=1,usecols="N:O",nrows=7)
data_cm2r=pd.read_excel(data_base,sheet_name='Data',skiprows=1,usecols="R:U",nrows=7)

c_nf54=pd.read_excel(data_base,sheet_name='Data',usecols="B:E",nrows=1).columns
c_fcr3=pd.read_excel(data_base,sheet_name='Data',usecols="F:I",nrows=1).columns
c_rbcs=pd.read_excel(data_base,sheet_name='Data',usecols="J:M",nrows=1).columns
c_biwo=pd.read_excel(data_base,sheet_name='Data',usecols="N:O",nrows=1).columns
c_cm2r=pd.read_excel(data_base,sheet_name='Data',usecols="R:U",nrows=1).columns

data_nf54.columns=c_nf54
data_fcr3.columns=c_fcr3
data_rbcs.columns=c_rbcs
data_biwo.columns=c_biwo
data_cm2r.columns=c_cm2r

tiempos=[0,1,5,15,30,45]
data_nf54.index=tiempos
data_fcr3.index=tiempos
data_rbcs.index=tiempos
data_biwo.index=tiempos
data_cm2r.index=tiempos

data_m_nf54=data_nf54.mean(axis=1)
data_m_fcr3=data_fcr3.mean(axis=1)
data_m_rbcs=data_rbcs.mean(axis=1)
data_m_biwo=data_biwo.mean(axis=1)
data_m_cm2r=data_cm2r.mean(axis=1)

def preparo_data(ddff=[]):
    x=[]
    y=[]
    d={}
    for i in ddff.columns:
      for qq,j in enumerate(ddff[i]):
        x.append(ddff[i].index.values[qq])
        y.append(j)
    d['x']=x
    d['y']=y
    return pd.Series(d['y'],index=d['x'])

data_ajuste_nf54=preparo_data(data_nf54).dropna()
data_ajuste_fcr3=preparo_data(data_fcr3).dropna()
data_ajuste_rbcs=preparo_data(data_rbcs).dropna()
data_ajuste_biwo=preparo_data(data_biwo).dropna()
data_ajuste_cm2r=preparo_data(data_cm2r).dropna()

In [ ]:
#Fitting of Model3

dt=0.02
#Functions
#---------------------------------------------------------

def calc_akaike(n, rss, k):
    """Calculates the Akaike Information Criterion (AIC)."""
    if (n - k - 1) <= 0:
        raise ValueError("The number of observations is too low")

    # AIC estándar
    aic = n * np.log(rss / n) + 2 * k

    # Término de corrección para muestras pequeñas/medianas
    correction = (2 * k * (k + 1)) / (n - k - 1)

    return aic + correction

def calc_rss(y_true, y_pred):
    """Calculates the Residual Sum of Squares"""
    return np.sum((np.array(y_true) - np.array(y_pred))**2)

# Model3
# ---------------------------------------------------------
def simulate_eATP_dynamics(t_eval, k_s_b, k_s_m, ATP_0, k_h, dt):
    """
    Euler integration for eATP dynamics.
    Calculates a high-resolution simulation and interpolates to match experimental times.
    """
    t_max = np.max(t_eval)
    tiempos = np.arange(0, t_max + dt, dt)

    ATP_sim = np.zeros_like(tiempos)
    ATP_sim[0] = ATP_0

    # Euler integration step
    for i in range(1, len(tiempos)):
        t = tiempos[i-1]
        k_salida = k_s_b + k_s_m * t
        dATPe_dt = k_salida - ATP_sim[i-1] * k_h
        ATP_sim[i] = ATP_sim[i-1] + dATPe_dt * dt

    # Interpolate to exactly match experimental time indices (t_eval)
    return np.interp(t_eval, tiempos, ATP_sim)

#Fittinh
# ---------------------------------------------------------
def fit_experimental_condition(name, data_series, k_h, dt,n_params=3):
    """Fits Model 3 to a specific condition and returns metrics."""
    print(f"--- Fitting Condition: {name} ---")
    print(f"Total Hydrolysis Constant: {k_h:.5f}")

    # Wrapper function locks in the specific hydrolysis constants for curve_fit
    def model_wrapper(t, k_s_b, k_s_m, ATP_0):
        return simulate_eATP_dynamics(t, k_s_b, k_s_m, ATP_0, k_h,dt)

    # Perform curve fitting
    popt, pcov = curve_fit(
        model_wrapper,
        data_series.index,
        data_series.values,
        p0=[0.01, 0.01, 2],
        bounds=([0, 0, 0], [10.0, 1.0, 60.0])
    )

    k_s_b, k_s_m, ATP_0 = popt
    err_k_s_b, err_k_s_m, err_ATP_0 = np.sqrt(np.diag(pcov))

    # Calculate performance metrics
    y_pred = model_wrapper(data_series.index, k_s_b, k_s_m, ATP_0)
    rss = calc_rss(data_series.values, y_pred)
    n_points = len(data_series.index)
    aic = calc_akaike(n_points, rss, n_params)

    print(f"Basal ATP efflux: {k_s_b:.5f} ± {err_k_s_b:.5f} nM/min")
    print(f"Incremental ATP efflux: {k_s_m:.5f} ± {err_k_s_m:.5f} nM/min²")
    print(f"Initial ATP (ATP_0): {ATP_0:.3f} ± {err_ATP_0:.3f} nM")
    print(f"RSS: {rss:.2f} | AIC: {aic:.2f}\n")

    # Format output specifically for the Supplementary Table
    supp_data = [
        f"{k_s_b:.3f} ± {err_k_s_b:.3f}",
        f"({k_s_m * 1000:.2f} ± {err_k_s_m * 1000:.2f})x10-3",
        f"{ATP_0:.2f} ± {err_ATP_0:.2f}",
        str(n_points),
        f"{rss:.2f}",
        f"{aic:.2f}"
    ]

    # Raw data for internal tracking/plotting
    raw_data = [k_s_b, k_s_m, ATP_0, rss, n_points, aic]

    return supp_data, raw_data

#Fitting to each dataset
#-----
conditions = [
    {'name': 'iRBCs NF54-CSA', 'data': data_ajuste_nf54, 'k_h': k_h_irbc+k_h_c},
    {'name': 'iRBCs FCR3-CSA', 'data': data_ajuste_fcr3, 'k_h': k_h_irbc+k_h_c},
    {'name': 'cmRBCs-FCR3',         'data': data_ajuste_rbcs, 'k_h': k_h_crbc+k_h_c},
    {'name': 'Mechanical stimulus', 'data': data_ajuste_biwo, 'k_h': +k_h_c},
    {'name': 'cmRBCs-NF54', 'data': data_ajuste_cm2r, 'k_h': +k_h_crbc+k_h_c}
]

# Initialize output dictionaries
ajustes_dict = {
    'Parameters': ['K salida basal', 'K salida ATP en función del tiempo', 'ATP0', 'Suma residuales', 'N', 'Akaike']
}
supplementary_dict = {
    'Data': ['Parameter kefflux-constant', 'Parameter kefflux-variable', 'Initial [eATP]', 'Data points', 'Residuals sum', 'Akaike information criterion']
}

# Run the fits iteratively
for cond in conditions:
    supp_res, raw_res = fit_experimental_condition(
        name=cond['name'],
        data_series=cond['data'],
        k_h=cond['k_h'],
        dt=dt
    )
    supplementary_dict[cond['name']] = supp_res
    ajustes_dict[cond['name']] = raw_res

ajustes_dict['Unidades'] = ['nM salida/min', 'nM salida/(min**2)', 'nM', '', '', '']

# ---------------------------------------------------------
# 5. Export Results (Using Local Relative Paths for GitHub)
# ---------------------------------------------------------
output_dir = base+'/Results'
os.makedirs(output_dir, exist_ok=True)

pd.DataFrame(ajustes_dict).to_csv(f'{output_dir}/Ajustes_Modelo_3_offline.csv', index=False)
pd.DataFrame(supplementary_dict).to_csv(f'{output_dir}/Supplementary_modelo3.csv', sep=';', decimal=',', index=False)

print("Fitting complete. Data saved to local directory.")

--- Fitting Condition: iRBCs NF54-CSA ---
Total Hydrolysis Constant: 0.01603
Basal ATP efflux: 0.01687 ± 0.00988 nM/min
Incremental ATP efflux: 0.00040 ± 0.00046 nM/min²
Initial ATP (ATP_0): 0.215 ± 0.063 nM
RSS: 0.67 | AIC: -69.51

--- Fitting Condition: iRBCs FCR3-CSA ---
Total Hydrolysis Constant: 0.01603
Basal ATP efflux: 0.01686 ± 0.02474 nM/min
Incremental ATP efflux: 0.00172 ± 0.00107 nM/min²
Initial ATP (ATP_0): 0.304 ± 0.166 nM
RSS: 5.12 | AIC: -29.86

--- Fitting Condition: cmRBCs-FCR3 ---
Total Hydrolysis Constant: 0.01469
Basal ATP efflux: 0.00000 ± 0.01485 nM/min
Incremental ATP efflux: 0.00103 ± 0.00064 nM/min²
Initial ATP (ATP_0): 0.262 ± 0.100 nM
RSS: 1.87 | AIC: -54.11

--- Fitting Condition: Mechanical stimulus ---
Total Hydrolysis Constant: 0.01448
Basal ATP efflux: 0.00600 ± 0.00439 nM/min
Incremental ATP efflux: 0.00014 ± 0.00019 nM/min²
Initial ATP (ATP_0): 0.320 ± 0.030 nM
RSS: 0.03 | AIC: -61.06

--- Fitting Condition: cmRBCs-NF54 ---
Total Hydrolysis Constant: 

In [ ]:
#Fitting of Model2


def simulate_eATP_dynamics(t_eval, k_s_m, ATP_0, k_h, dt):
    """
    Euler integration for eATP dynamics.
    Calculates a high-resolution simulation and interpolates to match experimental times.
    """
    t_max = np.max(t_eval)
    tiempos = np.arange(0, t_max + dt, dt)

    ATP_sim = np.zeros_like(tiempos)
    ATP_sim[0] = ATP_0

    # Euler integration step
    for i in range(1, len(tiempos)):
        t = tiempos[i-1]
        k_salida =  k_s_m * t
        dATPe_dt = k_salida - ATP_sim[i-1] * k_h
        ATP_sim[i] = ATP_sim[i-1] + dATPe_dt * dt

    # Interpolate to exactly match experimental time indices (t_eval)
    return np.interp(t_eval, tiempos, ATP_sim)

#Fittinh
# ---------------------------------------------------------
def fit_experimental_condition(name, data_series, k_h, dt,n_params=2):
    """Fits Model 3 to a specific condition and returns metrics."""
    print(f"--- Fitting Condition: {name} ---")
    print(f"Total Hydrolysis Constant: {k_h:.5f}")

    # Wrapper function locks in the specific hydrolysis constants for curve_fit
    def model_wrapper(t, k_s_m, ATP_0):
        return simulate_eATP_dynamics(t, k_s_m, ATP_0, k_h,dt)

    # Perform curve fitting
    popt, pcov = curve_fit(
        model_wrapper,
        data_series.index,
        data_series.values,
        p0=[ 0.01, 2],
        bounds=([0, 0], [1.0, 60.0])
    )

    k_s_m, ATP_0 = popt
    err_k_s_m, err_ATP_0 = np.sqrt(np.diag(pcov))

    # Calculate performance metrics
    y_pred = model_wrapper(data_series.index, k_s_m, ATP_0)
    rss = calc_rss(data_series.values, y_pred)
    n_points = len(data_series.index)
    aic = calc_akaike(n_points, rss, n_params)

    print(f"Incremental ATP efflux: {k_s_m:.5f} ± {err_k_s_m:.5f} nM/min²")
    print(f"Initial ATP (ATP_0): {ATP_0:.3f} ± {err_ATP_0:.3f} nM")
    print(f"RSS: {rss:.2f} | AIC: {aic:.2f}\n")

    # Format output specifically for the Supplementary Table
    supp_data = [
        f"({k_s_m * 1000:.2f} ± {err_k_s_m * 1000:.2f})x10-3",
        f"{ATP_0:.2f} ± {err_ATP_0:.2f}",
        str(n_points),
        f"{rss:.2f}",
        f"{aic:.2f}"
    ]

    # Raw data for internal tracking/plotting
    raw_data = [k_s_m, ATP_0, rss, n_points, aic]

    return supp_data, raw_data

#Fitting to each dataset
#-----
conditions = [
    {'name': 'iRBCs NF54-CSA', 'data': data_ajuste_nf54, 'k_h': k_h_irbc+k_h_c},
    {'name': 'iRBCs FCR3-CSA', 'data': data_ajuste_fcr3, 'k_h': k_h_irbc+k_h_c},
    {'name': 'cmRBCs-FCR3',         'data': data_ajuste_rbcs, 'k_h': k_h_crbc+k_h_c},
    {'name': 'Mechanical stimulus', 'data': data_ajuste_biwo, 'k_h': +k_h_c},
    {'name': 'cmRBCs-NF54',         'data': data_ajuste_cm2r, 'k_h': k_h_crbc+k_h_c}
]

# Initialize output dictionaries
ajustes_dict = {
    'Parameters': ['K salida ATP en función del tiempo', 'ATP0', 'Suma residuales', 'N', 'Akaike']
}
supplementary_dict = {
    'Data': ['Parameter kefflux-variable', 'Initial [eATP]', 'Data points', 'Residuals sum', 'Akaike information criterion']
}

# Run the fits iteratively
for cond in conditions:
    supp_res, raw_res = fit_experimental_condition(
        name=cond['name'],
        data_series=cond['data'],
        k_h=cond['k_h'],
        dt=dt
    )
    supplementary_dict[cond['name']] = supp_res
    ajustes_dict[cond['name']] = raw_res

ajustes_dict['Unidades'] = ['nM salida/(min**2)', 'nM', '', '', '']

# ---------------------------------------------------------
# 5. Export Results (Using Local Relative Paths for GitHub)
# ---------------------------------------------------------
output_dir = base+'/Results'
os.makedirs(output_dir, exist_ok=True)

pd.DataFrame(ajustes_dict).to_csv(f'{output_dir}/Ajustes_Modelo_2_offline.csv', index=False)
pd.DataFrame(supplementary_dict).to_csv(f'{output_dir}/Supplementary_modelo2.csv', sep=';', decimal=',', index=False)

print("Fitting complete. Data saved to local directory.")

--- Fitting Condition: iRBCs NF54-CSA ---
Total Hydrolysis Constant: 0.01603
Incremental ATP efflux: 0.00115 ± 0.00015 nM/min²
Initial ATP (ATP_0): 0.279 ± 0.054 nM
RSS: 0.77 | AIC: -69.06

--- Fitting Condition: iRBCs FCR3-CSA ---
Total Hydrolysis Constant: 0.01603
Incremental ATP efflux: 0.00242 ± 0.00030 nM/min²
Initial ATP (ATP_0): 0.372 ± 0.132 nM
RSS: 5.24 | AIC: -31.97

--- Fitting Condition: cmRBCs-FCR3 ---
Total Hydrolysis Constant: 0.01469
Incremental ATP efflux: 0.00103 ± 0.00017 nM/min²
Initial ATP (ATP_0): 0.262 ± 0.078 nM
RSS: 1.87 | AIC: -56.74

--- Fitting Condition: Mechanical stimulus ---
Total Hydrolysis Constant: 0.01448
Incremental ATP efflux: 0.00039 ± 0.00006 nM/min²
Initial ATP (ATP_0): 0.344 ± 0.025 nM
RSS: 0.04 | AIC: -62.46

--- Fitting Condition: cmRBCs-NF54 ---
Total Hydrolysis Constant: 0.01469
Incremental ATP efflux: 0.00064 ± 0.00008 nM/min²
Initial ATP (ATP_0): 0.288 ± 0.027 nM
RSS: 0.20 | AIC: -99.27

Fitting complete. Data saved to local directory.


In [ ]:
#Fitting of Model1


def simulate_eATP_dynamics(t_eval, k_s_b, ATP_0, k_h, dt):
    """
    Euler integration for eATP dynamics.
    Calculates a high-resolution simulation and interpolates to match experimental times.
    """
    t_max = np.max(t_eval)
    tiempos = np.arange(0, t_max + dt, dt)

    ATP_sim = np.zeros_like(tiempos)
    ATP_sim[0] = ATP_0

    # Euler integration step
    for i in range(1, len(tiempos)):
        t = tiempos[i-1]
        k_salida =  k_s_b
        dATPe_dt = k_salida - ATP_sim[i-1] * k_h
        ATP_sim[i] = ATP_sim[i-1] + dATPe_dt * dt

    # Interpolate to exactly match experimental time indices (t_eval)
    return np.interp(t_eval, tiempos, ATP_sim)

#Fittinh
# ---------------------------------------------------------
def fit_experimental_condition(name, data_series, k_h, dt,n_params=2):
    """Fits Model 3 to a specific condition and returns metrics."""
    print(f"--- Fitting Condition: {name} ---")
    print(f"Total Hydrolysis Constant: {k_h:.5f}")

    # Wrapper function locks in the specific hydrolysis constants for curve_fit
    def model_wrapper(t, k_s_b, ATP_0):
        return simulate_eATP_dynamics(t, k_s_b, ATP_0, k_h,dt)

    # Perform curve fitting
    popt, pcov = curve_fit(
        model_wrapper,
        data_series.index,
        data_series.values,
        p0=[ 0.01, 2],
        bounds=([0, 0], [10.0, 60.0])
    )

    k_s_b, ATP_0 = popt
    err_k_s_b, err_ATP_0 = np.sqrt(np.diag(pcov))

    # Calculate performance metrics
    y_pred = model_wrapper(data_series.index, k_s_b, ATP_0)
    rss = calc_rss(data_series.values, y_pred)
    n_points = len(data_series.index)
    aic = calc_akaike(n_points, rss, n_params)

    print(f"Fixed ATP efflux: {k_s_b:.5f} ± {err_k_s_b:.5f} nM/min")
    print(f"Initial ATP (ATP_0): {ATP_0:.3f} ± {err_ATP_0:.3f} nM")
    print(f"RSS: {rss:.2f} | AIC: {aic:.2f}\n")

    # Format output specifically for the Supplementary Table
    supp_data = [
        f"({k_s_b:.4f} ± {err_k_s_b:.4f})",
        f"{ATP_0:.2f} ± {err_ATP_0:.2f}",
        str(n_points),
        f"{rss:.2f}",
        f"{aic:.2f}"
    ]

    # Raw data for internal tracking/plotting
    raw_data = [k_s_b, ATP_0, rss, n_points, aic]

    return supp_data, raw_data

#Fitting to each dataset
#-----
conditions = [
    {'name': 'iRBCs NF54-CSA', 'data': data_ajuste_nf54, 'k_h': k_h_irbc+k_h_c},
    {'name': 'iRBCs FCR3-CSA', 'data': data_ajuste_fcr3, 'k_h': k_h_irbc+k_h_c},
    {'name': 'cmRBCs-FCR3',         'data': data_ajuste_rbcs, 'k_h': k_h_crbc+k_h_c},
    {'name': 'Mechanical stimulus', 'data': data_ajuste_biwo, 'k_h': +k_h_c},
    {'name': 'cmRBCs-NF54',         'data': data_ajuste_cm2r, 'k_h': k_h_crbc+k_h_c},
]

# Initialize output dictionaries
ajustes_dict = {
    'Parameters': ['ATP en función del tiempo', 'ATP0', 'Suma residuales', 'N', 'Akaike']
}
supplementary_dict = {
    'Data': ['Parameter kefflux', 'Initial [eATP]', 'Data points', 'Residuals sum', 'Akaike information criterion']
}

# Run the fits iteratively
for cond in conditions:
    supp_res, raw_res = fit_experimental_condition(
        name=cond['name'],
        data_series=cond['data'],
        k_h=cond['k_h'],
        dt=dt
    )
    supplementary_dict[cond['name']] = supp_res
    ajustes_dict[cond['name']] = raw_res

ajustes_dict['Unidades'] = ['nM salida/min', 'nM', '', '', '']

# ---------------------------------------------------------
# 5. Export Results (Using Local Relative Paths for GitHub)
# ---------------------------------------------------------
output_dir = base+'/Results'
os.makedirs(output_dir, exist_ok=True)

pd.DataFrame(ajustes_dict).to_csv(f'{output_dir}/Ajustes_Modelo_1_offline.csv', index=False)
pd.DataFrame(supplementary_dict).to_csv(f'{output_dir}/Supplementary_modelo1.csv', sep=';', decimal=',', index=False)

print("Fitting complete. Data saved to local directory.")

--- Fitting Condition: iRBCs NF54-CSA ---
Total Hydrolysis Constant: 0.01603
Fixed ATP efflux: 0.02505 ± 0.00309 nM/min
Initial ATP (ATP_0): 0.190 ± 0.056 nM
RSS: 0.70 | AIC: -71.34

--- Fitting Condition: iRBCs FCR3-CSA ---
Total Hydrolysis Constant: 0.01603
Fixed ATP efflux: 0.05516 ± 0.00718 nM/min
Initial ATP (ATP_0): 0.176 ± 0.151 nM
RSS: 5.76 | AIC: -29.69

--- Fitting Condition: cmRBCs-FCR3 ---
Total Hydrolysis Constant: 0.01469
Fixed ATP efflux: 0.02256 ± 0.00434 nM/min
Initial ATP (ATP_0): 0.188 ± 0.092 nM
RSS: 2.16 | AIC: -53.18

--- Fitting Condition: Mechanical stimulus ---
Total Hydrolysis Constant: 0.01448
Fixed ATP efflux: 0.00920 ± 0.00119 nM/min
Initial ATP (ATP_0): 0.309 ± 0.025 nM
RSS: 0.04 | AIC: -63.98

--- Fitting Condition: cmRBCs-NF54 ---
Total Hydrolysis Constant: 0.01469
Fixed ATP efflux: 0.01392 ± 0.00152 nM/min
Initial ATP (ATP_0): 0.237 ± 0.028 nM
RSS: 0.17 | AIC: -102.04

Fitting complete. Data saved to local directory.


In [ ]:
#Fitting of Model4

dt=0.02
#Functions
#---------------------------------------------------------
def calc_rss(y_true, y_pred):
    """Calculates the Residual Sum of Squares (Suma Rústica)."""
    return np.sum((np.array(y_true) - np.array(y_pred))**2)

# Model3
# ---------------------------------------------------------
def simulate_eATP_dynamics(t_eval, k_s_A, k_s_exp, ATP_0, k_h, dt):
    """
    Euler integration for eATP dynamics.
    Calculates a high-resolution simulation and interpolates to match experimental times.
    """
    t_max = np.max(t_eval)
    tiempos = np.arange(0, t_max + dt, dt)

    ATP_sim = np.zeros_like(tiempos)
    ATP_sim[0] = ATP_0

    # Euler integration step
    for i in range(1, len(tiempos)):
        t = tiempos[i-1]
        k_salida = k_s_A*(1-np.exp(-k_s_exp * t))
        dATPe_dt = k_salida - ATP_sim[i-1] * k_h
        ATP_sim[i] = ATP_sim[i-1] + dATPe_dt * dt

    # Interpolate to exactly match experimental time indices (t_eval)
    return np.interp(t_eval, tiempos, ATP_sim)

#Fittinh
# ---------------------------------------------------------
def fit_experimental_condition(name, data_series, k_h, dt,n_params=3):
    """Fits Model 3 to a specific condition and returns metrics."""
    print(f"--- Fitting Condition: {name} ---")
    print(f"Total Hydrolysis Constant: {k_h:.5f}")

    # Wrapper function locks in the specific hydrolysis constants for curve_fit
    def model_wrapper(t, k_s_A, k_s_exp, ATP_0):
        return simulate_eATP_dynamics(t, k_s_A, k_s_exp, ATP_0, k_h,dt)

    # Perform curve fitting
    popt, pcov = curve_fit(
        model_wrapper,
        data_series.index,
        data_series.values,
        p0=[0.2, 0.001, 0.2],
        bounds=([0.00001, 0.00001, 0], [10, 0.5, 1.0])
    )

    k_s_A, k_s_exp, ATP_0 = popt
    err_k_s_A, err_k_s_exp, err_ATP_0 = np.sqrt(np.diag(pcov))

    # Calculate performance metrics
    y_pred = model_wrapper(data_series.index, k_s_A, k_s_exp, ATP_0)
    rss = calc_rss(data_series.values, y_pred)
    n_points = len(data_series.index)
    aic = calc_akaike(n_points, rss, n_params)

    print(f"Maximal ATP efflux: {k_s_A:.5f} ± {err_k_s_exp:.5f} nM/min")
    print("ATP efflux growth kinetics: "+ str(k_s_exp)+" ± "+str(err_k_s_exp))
    print(f"Initial ATP (ATP_0): {ATP_0:.3f} ± {err_ATP_0:.3f} nM")
    print(f"RSS: {rss:.2f} | AIC: {aic:.2f}\n")

    # Format output specifically for the Supplementary Table
    supp_data = [
        f"{k_s_A:.3f} ± {err_k_s_A:.3f}",
        f"({k_s_exp:.2f} ± {err_k_s_exp:.2f})",
        f"{ATP_0:.2f} ± {err_ATP_0:.2f}",
        str(n_points),
        f"{rss:.2f}",
        f"{aic:.2f}"
    ]

    # Raw data for internal tracking/plotting
    raw_data = [k_s_A, k_s_exp, ATP_0, rss, n_points, aic]

    return supp_data, raw_data

#Fitting to each dataset
#-----
conditions = [
    {'name': 'iRBCs NF54-CSA', 'data': data_ajuste_nf54, 'k_h': k_h_irbc+k_h_c},
    {'name': 'iRBCs FCR3-CSA', 'data': data_ajuste_fcr3, 'k_h': k_h_irbc+k_h_c},
    {'name': 'cmRBCs-FCR3',         'data': data_ajuste_rbcs, 'k_h': k_h_crbc+k_h_c},
    {'name': 'Mechanical stimulus', 'data': data_ajuste_biwo, 'k_h': +k_h_c},
    {'name': 'cmRBCs-NF54',         'data': data_ajuste_cm2r, 'k_h': k_h_crbc+k_h_c},
]

# Initialize output dictionaries
ajustes_dict = {
    'Parameters': ['Máxima salida', 'Aumento de salida ATP', 'ATP0', 'Suma residuales', 'N', 'Akaike']
}
supplementary_dict = {
    'Data': ['Maximal kefflux', 'kefflux growth', 'Initial [eATP]', 'Data points', 'Residuals sum', 'Akaike information criterion']
}

# Run the fits iteratively
for cond in conditions:
    supp_res, raw_res = fit_experimental_condition(
        name=cond['name'],
        data_series=cond['data'],
        k_h=cond['k_h'],
        dt=dt
    )
    supplementary_dict[cond['name']] = supp_res
    ajustes_dict[cond['name']] = raw_res

ajustes_dict['Unidades'] = ['nM salida/min', '1/min', 'nM', '', '', '']

# ---------------------------------------------------------
# 5. Export Results (Using Local Relative Paths for GitHub)
# ---------------------------------------------------------
output_dir = base+'/Results'
os.makedirs(output_dir, exist_ok=True)
print(output_dir)
pd.DataFrame(ajustes_dict).to_csv(f'{output_dir}/Ajustes_Modelo_4_offline.csv', index=False)
pd.DataFrame(supplementary_dict).to_csv(f'{output_dir}/Supplementary_modelo4.csv', sep=';', decimal=',', index=False)

print("Fitting complete. Data saved to local directory.")

--- Fitting Condition: iRBCs NF54-CSA ---
Total Hydrolysis Constant: 0.01603
Maximal ATP efflux: 0.03051 ± 0.15672 nM/min
ATP efflux growth kinetics: 0.12713270015575762 ± 0.15671972576678397
Initial ATP (ATP_0): 0.238 ± 0.060 nM
RSS: 0.67 | AIC: -69.59

--- Fitting Condition: iRBCs FCR3-CSA ---
Total Hydrolysis Constant: 0.01603
Maximal ATP efflux: 0.10802 ± 0.06925 nM/min
ATP efflux growth kinetics: 0.03622133314185901 ± 0.06925371030393777
Initial ATP (ATP_0): 0.335 ± 0.148 nM
RSS: 5.14 | AIC: -29.77

--- Fitting Condition: cmRBCs-FCR3 ---
Total Hydrolysis Constant: 0.01469
Maximal ATP efflux: 9.99991 ± 0.07234 nM/min
ATP efflux growth kinetics: 0.00010319916543736023 ± 0.07234107286088229
Initial ATP (ATP_0): 0.262 ± 0.087 nM
RSS: 1.87 | AIC: -54.11

--- Fitting Condition: Mechanical stimulus ---
Total Hydrolysis Constant: 0.01448
Maximal ATP efflux: 0.01083 ± 0.19679 nM/min
ATP efflux growth kinetics: 0.13444148260680902 ± 0.1967949362184792
Initial ATP (ATP_0): 0.327 ± 0.027 nM
R

In [ ]:
import os
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Simulation model 2
# ---------------------------------------------------------

def calc_flux(t_max, k_s_m, dt=0.1):
    """Jefflux = k_s_m * t."""
    tiempos = np.arange(0, t_max + dt, dt)
    flux = k_s_m * tiempos
    return tiempos, flux

def simulate_no_hydrolysis(t_max, k_s_m, ATP_0, dt=0.1):
    """
    [eATP] simulation assuming no hydrolysis (k_h = 0).
    Analytical solution: [eATP](t) = ATP_0 + 0.5 * k_s_m * t^2
    """
    tiempos = np.arange(0, t_max + dt, dt)
    eATP_no_hydrolysis = ATP_0 + 0.5 * k_s_m * (tiempos ** 2)
    return tiempos, eATP_no_hydrolysis

def simulate_model2_full(t_max, k_s_m, ATP_0, k_h_total, dt=0.1):
    """Euler integration Model 2"""
    tiempos = np.arange(0, t_max + dt, dt)
    eATP_sim = np.zeros_like(tiempos)
    eATP_sim[0] = ATP_0

    for i in range(1, len(tiempos)):
        t = tiempos[i-1]
        dATP_dt = (k_s_m * t) - eATP_sim[i-1] * k_h_total
        eATP_sim[i] = eATP_sim[i-1] + dATP_dt * dt
    return tiempos, eATP_sim

def simulate_model1_full(t_max, k_s_b, ATP_0, k_h_total, dt=0.1):
    """Euler integration Model 2"""
    tiempos = np.arange(0, t_max + dt, dt)
    eATP_sim = np.zeros_like(tiempos)
    eATP_sim[0] = ATP_0

    for i in range(1, len(tiempos)):
        t = tiempos[i-1]
        dATP_dt = k_s_b - eATP_sim[i-1] * k_h_total
        eATP_sim[i] = eATP_sim[i-1] + dATP_dt * dt
    return tiempos, eATP_sim

input_path = base+'/Results/Ajustes_Modelo_2_offline.csv'
df_params = pd.read_csv(input_path)
print(df_params)

# Extract parameters
def get_model_params(df, condition_name):
    k_s_m = float(df[condition_name].iloc[0])
    ATP_0 = float(df[condition_name].iloc[1])
    return k_s_m, ATP_0

# Simulations of iRBCs-FCR3 and cRBCs

output_dir = base+'/Simulations'
os.makedirs(output_dir, exist_ok=True)

# Maximal simulation time
t_max_dict = {
    'iRBCs FCR3-CSA': 45,
    'uiRBCs': 45,
}

# File names
conditions_map = [
    {'csv_key': 'iRBCs FCR3-CSA', 'filename': 'simulation_FCR3.csv'},
    {'csv_key': 'cmRBCs-FCR3', 'filename': 'simulation_cmRBCs_FCR3.csv'},
]

# Simulations of each condition
for cond in conditions_map:
    key = cond['csv_key']
    k_s_m, ATP_0 = get_model_params(df_params, key)
    t_max = 45.0  # Maximal time

    # Calculation of flux and eATP without hydrolysis
    t_flux, y_flux = calc_flux(t_max, k_s_m)
    t_out, y_out = simulate_no_hydrolysis(t_max, k_s_m, ATP_0)

    # Gather data
    df_sim = pd.DataFrame({
        'time_flux_min': t_flux,
        'Jefflux_nM_per_min': y_flux,
        'time_eATP_min': t_out,
        'eATP_no_hydrolysis_nM': y_out
    })

    # Export file
    df_sim.to_csv(f'{output_dir}/{cond["filename"]}', index=False)
    print(f"Guardado: {output_dir}/{cond['filename']}")


input_path = base+'/Results/Ajustes_Modelo_2_offline.csv'
df_params = pd.read_csv(input_path)
print(df_params)

# Extract parameters
def get_model_params(df, condition_name):
    k_s_b = float(df[condition_name].iloc[0])
    ATP_0 = float(df[condition_name].iloc[1])
    return k_s_b, ATP_0

# Simulations

output_dir = base+'/Simulations'
os.makedirs(output_dir, exist_ok=True)

# Maximal simulation time
t_max_dict = {
    'iRBCs NF54-CSA': 45,
    'Mechanical stimulus': 45
}

# File names
conditions_map = [
    {'csv_key': 'iRBCs NF54-CSA', 'filename': 'simulation_NF54.csv'},
    {'csv_key': 'Mechanical stimulus', 'filename': 'simulation_BiWo_mechanical.csv'},
    {'csv_key': 'cmRBCs-NF54', 'filename': 'simulation_cmRBCS_NF54.csv'}
]

# Simulations of each condition
for cond in conditions_map:
    key = cond['csv_key']
    k_s_m, ATP_0 = get_model_params(df_params, key)
    t_max = 45.0  # Maximal time

    # Calculation of flux and eATP without hydrolysis
    t_flux, y_flux = calc_flux(t_max, k_s_m)

    # Gather data
    df_sim = pd.DataFrame({
        'time_flux_min': t_flux,
        'Jefflux_nM_per_min': y_flux,
        'time_eATP_min': t_out,
        'eATP_no_hydrolysis_nM': y_out
    })

    # Export file
    df_sim.to_csv(f'{output_dir}/{cond["filename"]}', index=False)
    print(f"Guardado: {output_dir}/{cond['filename']}")


# Placental explants simulations
k_s_m_fcr3, ATP_0_fcr3 = get_model_params(df_params, 'iRBCs FCR3-CSA')
k_h_explantes = 0.0582
t_max_expl = 45.0

# 1. Hydrolysis simulation
t_expl, y_expl = simulate_model2_full(t_max_expl, k_s_m_fcr3, ATP_0_fcr3, k_h_explantes)
df_expl = pd.DataFrame({
    'time_min': t_expl,
    'eATP_explants_nM': y_expl
})
df_expl.to_csv(f'{output_dir}/simulation_explants_with_hydrolysis.csv', index=False)
print(f"Guardado: {output_dir}/simulation_explants_with_hydrolysis.csv")

# 2. Simulation without hydrolysis
t_out_expl, y_out_expl = simulate_no_hydrolysis(t_max_expl, k_s_m_fcr3, ATP_0_fcr3)
df_expl_no_lysis = pd.DataFrame({
    'time_min': t_out_expl,
    'eATP_explants_no_hydrolysis_nM': y_out_expl
})
df_expl_no_lysis.to_csv(f'{output_dir}/simulation_explants_no_hydrolysis.csv', index=False)
print(f"Guardado: {output_dir}/simulation_explants_no_hydrolysis.csv")

print("\nSimulations completed successfully.")

                           Parameters  iRBCs NF54-CSA  iRBCs FCR3-CSA  \
0  K salida ATP en función del tiempo        0.001153        0.002421   
1                                ATP0        0.279375        0.371669   
2                     Suma residuales        0.772007        5.236444   
3                                   N       22.000000       24.000000   
4                              Akaike      -69.064123      -31.966440   

   cmRBCs-FCR3  Mechanical stimulus  cmRBCs-NF54            Unidades  
0     0.001030             0.000394     0.000642  nM salida/(min**2)  
1     0.261745             0.344324     0.287542                  nM  
2     1.865148             0.042225     0.195607                 NaN  
3    24.000000            12.000000    22.000000                 NaN  
4   -56.741689           -62.462575   -99.267606                 NaN  
Guardado: /content/drive/MyDrive/Modelo/eATP-kinetics-model/Simulations/simulation_FCR3.csv
Guardado: /content/drive/MyDrive/Modelo/eAT